## Embedding Generation

This notebook loads the preprocessed dataset from Notebook 1 and generates sentence embeddings for all 30 model-configuration combinations: five multilingual and monolingual models across six text configurations each. All embeddings are L2 normalized before saving so that cosine similarity calculations in Notebook 3 reduce to a simple dot product on unit vectors.

### Imports and Setup

In [1]:
import sys
import time
import random
import warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from transformers import logging as hf_logging

# Reproducibility: set all relevant seeds. Inference is largely deterministic,
# but CUDA kernels and tokenizer shuffling can still introduce minor variation
# across runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Narrow warnings suppression: specific known-benign categories only,
# so unexpected warnings remain visible.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sentence_transformers")
hf_logging.set_verbosity_error()

print(f"Python               : {sys.version.split()[0]}")
print(f"torch                : {torch.__version__}")
print(f"transformers         : {version('transformers')}")
print(f"sentence-transformers: {version('sentence-transformers')}")
print(f"numpy                : {np.__version__}")
print(f"pandas               : {pd.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device          : {torch.cuda.get_device_name(0)}")

Python               : 3.10.20
torch                : 2.12.0.dev20260408+cu128
transformers         : 5.4.0
sentence-transformers: 5.3.0
numpy                : 2.2.6
pandas               : 2.3.3
CUDA available       : True
CUDA device          : NVIDIA GeForce RTX 5060 Laptop GPU


### Project Paths

All paths are derived from the notebook's working directory so that the pipeline runs unchanged on any machine. The notebook assumes it is executed either from the project root or from a `notebooks/` subdirectory.

In [2]:
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = DATA_DIR / "embeddings" / "ict"

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root     : {PROJECT_ROOT}")
print(f"Processed data   : {PROCESSED_DIR}")
print(f"Embeddings out   : {EMBEDDINGS_DIR}")

Project root     : c:\madee\thesis-project
Processed data   : c:\madee\thesis-project\data\processed
Embeddings out   : c:\madee\thesis-project\data\embeddings\ict


### Loading the Preprocessed Dataset

The preprocessed dataset produced by Notebook 1 is loaded here. All twelve configuration columns are available and ready for embedding generation.

In [3]:
data_path = PROCESSED_DIR / "dataset_preprocessed.csv"
df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape}")
print(f"\nLabel distribution:\n{df['similarity_label'].value_counts()}")

config_cols = [c for c in df.columns if c.startswith('config_')]
print(f"\n{len(config_cols)} configuration columns available:")
for col in config_cols:
    print(f"  {col}")

Dataset loaded: (154, 33)

Label distribution:
similarity_label
1    77
0    77
Name: count, dtype: int64

12 configuration columns available:
  config_1_fi
  config_1_en
  config_2_fi
  config_2_en
  config_3_fi
  config_3_en
  config_4_fi
  config_4_en
  config_5_fi
  config_5_en
  config_6_fi
  config_6_en


### Defining the Five Models

Five models are evaluated. Four are multilingual sentence embedding models representing different architectural and training strategies, and the fifth is FinBERT, a Finnish monolingual model included as a negative baseline. LaBSE was added after initial evaluation because exploratory results showed it outperforming the first four models on this task.

| Model key | Model name                                          | Type          |
|-----------|-----------------------------------------------------|---------------|
| mpnet     | paraphrase-multilingual-mpnet-base-v2               | Multilingual  |
| stsb      | stsb-xlm-r-multilingual                             | Multilingual  |
| e5        | intfloat/multilingual-e5-base                       | Multilingual  |
| labse     | sentence-transformers/LaBSE                         | Multilingual  |
| finbert   | TurkuNLP/bert-base-finnish-cased-v1                 | Monolingual   |

All models share a maximum input length of 512 subword tokens. This value is set explicitly on each model after loading so that longer configurations use the full available context rather than falling back to smaller model defaults.

In [4]:
models = {
    'mpnet'  : 'paraphrase-multilingual-mpnet-base-v2',
    'stsb'   : 'stsb-xlm-r-multilingual',
    'e5'     : 'intfloat/multilingual-e5-base',
    'labse'  : 'sentence-transformers/LaBSE',
    'finbert': 'TurkuNLP/bert-base-finnish-cased-v1',
}

MAX_SEQ_LEN = 512
BATCH_SIZE = 32

print("Models defined:")
for key, name in models.items():
    print(f"  {key:<8}: {name}")
print(f"\nMaximum sequence length : {MAX_SEQ_LEN}")
print(f"Batch size              : {BATCH_SIZE}")

Models defined:
  mpnet   : paraphrase-multilingual-mpnet-base-v2
  stsb    : stsb-xlm-r-multilingual
  e5      : intfloat/multilingual-e5-base
  labse   : sentence-transformers/LaBSE
  finbert : TurkuNLP/bert-base-finnish-cased-v1

Maximum sequence length : 512
Batch size              : 32


### FinBERT as a Negative Baseline

FinBERT (Virtanen et al., 2019) is a Finnish monolingual BERT model trained by TurkuNLP on large Finnish text corpora. It is included here specifically as a negative baseline. Because FinBERT is not a cross-lingual model, its English-side embeddings are expected to be semantically unreliable, and the resulting cross-lingual cosine similarity scores against Finnish embeddings are expected to degrade notably relative to the multilingual models. This comparison serves to empirically demonstrate the necessity of multilingual models for bilingual course description matching, consistent with the cross-lingual transfer results reported by Reimers and Gurevych (2020) and Conneau et al. (2020).

FinBERT does not ship with a sentence-level pooling head, so a mean pooling strategy is applied manually over the final-layer token embeddings, masking out padding tokens through the attention mask. This is the canonical mean pooling scheme established by Reimers and Gurevych (2019) for adapting general-purpose BERT models to sentence-level similarity tasks.

### e5 Prefix Convention

The multilingual-e5-base model expects each input text to begin with either a `query: ` or a `passage: ` prefix. The official model documentation specifies that for symmetric tasks such as semantic textual similarity, bitext mining, and paraphrase retrieval, both inputs should use the `query: ` prefix, while the `query: ` and `passage: ` pair is reserved for asymmetric retrieval tasks such as open-domain question answering.

Bilingual course description matching is a symmetric task: both the Finnish and the English descriptions are documents of equivalent type, and neither side functions as a query against the other. The `query: ` prefix is therefore applied to both sides, following the model authors' recommendation. Empirical testing reported by the model authors indicates a small but consistent advantage for the `query: ` prefix over the `passage: ` prefix on symmetric STS tasks.

### Embedding Functions

Two encoding paths are used. The four multilingual models go through the sentence-transformers `.encode()` API, which handles batching, tokenization, pooling, and attention masking internally. FinBERT goes through a custom batched encoder that applies mean pooling manually. The FinBERT encoder processes texts in batches rather than one at a time, so each forward pass through the model covers many texts simultaneously and the GPU-to-CPU transfer happens once per batch rather than once per text.

All embeddings are L2 normalized after generation. The L2 normalization step is verified through an assertion that halts execution if any produced vector deviates from unit length by more than 1e-5.

In [5]:
def encode_sentence_transformer(model, texts, batch_size=BATCH_SIZE):
    """Encode texts using a sentence-transformers model."""
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

def encode_finbert(tokenizer, model, texts, device, batch_size=BATCH_SIZE, max_length=MAX_SEQ_LEN):
    """Encode texts with FinBERT using batched mean pooling over token embeddings.

    The attention mask is used to exclude padding tokens from the mean, which
    is the canonical mean pooling scheme established by Reimers and Gurevych (2019).
    Batching keeps tokenization, forward pass, pooling, and the GPU-CPU transfer
    at O(n_batches) rather than O(n_texts).
    """
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(
                batch,
                return_tensors='pt',
                truncation=True,
                max_length=max_length,
                padding=True,
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            token_emb = outputs.last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1).expand(token_emb.size()).float()
            summed = torch.sum(token_emb * mask, dim=1)
            denom = torch.clamp(mask.sum(dim=1), min=1e-9)
            batch_emb = (summed / denom).cpu().numpy()
            all_embeddings.append(batch_emb)
    return np.vstack(all_embeddings)

def l2_normalize(embeddings):
    """Normalize each row to unit L2 norm."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / np.clip(norms, a_min=1e-10, a_max=None)

print("Encoding functions defined:")
print("  encode_sentence_transformer : mpnet, stsb, e5, labse")
print("  encode_finbert              : FinBERT with batched manual mean pooling")
print("  l2_normalize                : applied to every embedding before saving")

Encoding functions defined:
  encode_sentence_transformer : mpnet, stsb, e5, labse
  encode_finbert              : FinBERT with batched manual mean pooling
  l2_normalize                : applied to every embedding before saving


### Generating and Saving Embeddings

Embeddings are generated for all 30 model-configuration combinations. For each combination, Finnish and English texts are encoded separately, L2 normalized, verified, and saved as NumPy files, producing 60 files in total.

Two auxiliary records are built during the loop. The first records how long each model takes to load and to encode all six configurations, and is used in the methodology chapter to discuss computational cost. The second records, per model and per configuration, how many of the 154 texts exceed 512 subword tokens before truncation. The second record directly supports the discussion of the 512-token truncation limitation.

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

configs = [1, 2, 3, 4, 5, 6]
languages = ['fi', 'en']

timing_records = []
trunc_records = []
tokenizer = None  # initialised here so the cleanup block can delete it uniformly

for model_key, model_name in models.items():
    print(f"Loading {model_key}: {model_name}")
    t0 = time.perf_counter()

    if model_key == 'finbert':
        tok = AutoTokenizer.from_pretrained(model_name)
        model_obj = AutoModel.from_pretrained(model_name).to(device)
        tokenizer = tok
    else:
        model_obj = SentenceTransformer(model_name, device=str(device))
        model_obj.max_seq_length = MAX_SEQ_LEN
        tok = model_obj.tokenizer
        print(f"  max_seq_length set to {model_obj.max_seq_length}")

    load_time = time.perf_counter() - t0
    print(f"  load time: {load_time:.1f}s")

    t1 = time.perf_counter()
    for config_num in configs:
        for lang in languages:
            col = f'config_{config_num}_{lang}'
            texts = df[col].tolist()

            # e5 uses the "query: " prefix on both sides for symmetric STS
            if model_key == 'e5':
                texts = [f'query: {t}' for t in texts]

            # Truncation tracking on the exact text that will be encoded
            token_lens = np.array([
                len(tok.encode(t, add_special_tokens=True, truncation=False))
                for t in texts
            ])
            trunc_records.append({
                'model'                 : model_key,
                'config'                : config_num,
                'language'              : lang,
                'n_texts'               : len(texts),
                'mean_tokens'           : float(token_lens.mean()),
                'median_tokens'         : float(np.median(token_lens)),
                'max_tokens_untruncated': int(token_lens.max()),
                'n_truncated'           : int((token_lens > MAX_SEQ_LEN).sum()),
                'pct_truncated'         : float(100 * (token_lens > MAX_SEQ_LEN).mean()),
            })

            # Encode
            if model_key == 'finbert':
                emb = encode_finbert(tokenizer, model_obj, texts, device)
            else:
                emb = encode_sentence_transformer(model_obj, texts)

            # L2 normalize
            emb = l2_normalize(emb)

            # Verify shape and normalization
            assert emb.shape == (len(df), 768), (
                f"Unexpected shape for {model_key} config {config_num} {lang}: {emb.shape}"
            )
            norms = np.linalg.norm(emb, axis=1)
            assert np.allclose(norms, 1.0, atol=1e-5), (
                f"L2 normalization failed for {model_key} config {config_num} {lang}: "
                f"norms range [{norms.min():.6f}, {norms.max():.6f}]"
            )

            # Save
            out_path = EMBEDDINGS_DIR / f"{model_key}_config{config_num}_{lang}.npy"
            np.save(out_path, emb)

        print(f"  config {config_num}: FI + EN done")

    encode_time = time.perf_counter() - t1
    timing_records.append({
        'model'          : model_key,
        'load_time_sec'  : load_time,
        'encode_time_sec': encode_time,
        'total_time_sec' : load_time + encode_time,
    })
    print(f"  {model_key} complete in {encode_time:.1f}s (encode) + {load_time:.1f}s (load)\n")

    del model_obj
    del tokenizer
    tokenizer = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("All 60 embedding files saved.")

Using device: cuda

Loading mpnet: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  max_seq_length set to 512
  load time: 23.9s
  config 1: FI + EN done
  config 2: FI + EN done
  config 3: FI + EN done
  config 4: FI + EN done
  config 5: FI + EN done
  config 6: FI + EN done
  mpnet complete in 31.3s (encode) + 23.9s (load)

Loading stsb: stsb-xlm-r-multilingual


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  max_seq_length set to 512
  load time: 25.9s
  config 1: FI + EN done
  config 2: FI + EN done
  config 3: FI + EN done
  config 4: FI + EN done
  config 5: FI + EN done
  config 6: FI + EN done
  stsb complete in 30.4s (encode) + 25.9s (load)

Loading e5: intfloat/multilingual-e5-base


...

  max_seq_length set to 512
  load time: 14.4s
  config 1: FI + EN done
  config 2: FI + EN done
  config 3: FI + EN done
  config 4: FI + EN done
  config 5: FI + EN done
  config 6: FI + EN done
  e5 complete in 31.0s (encode) + 14.4s (load)

Loading labse: sentence-transformers/LaBSE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  max_seq_length set to 512
  load time: 38.5s
  config 1: FI + EN done
  config 2: FI + EN done
  config 3: FI + EN done
  config 4: FI + EN done
  config 5: FI + EN done
  config 6: FI + EN done
  labse complete in 29.5s (encode) + 38.5s (load)

Loading finbert: TurkuNLP/bert-base-finnish-cased-v1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  load time: 14.6s
  config 1: FI + EN done
  config 2: FI + EN done
  config 3: FI + EN done
  config 4: FI + EN done
  config 5: FI + EN done
  config 6: FI + EN done
  finbert complete in 38.7s (encode) + 14.6s (load)

All 60 embedding files saved.


### Timing Summary

Per-model load and encoding times are displayed and saved as a CSV for reference in the methodology chapter. Encoding time is the wall-clock time to process all six configurations across both languages.

In [7]:
timing_df = pd.DataFrame(timing_records)
timing_df = timing_df.round(2)
print(timing_df.to_string(index=False))

timing_path = PROCESSED_DIR / "embedding_timing.csv"
timing_df.to_csv(timing_path, index=False)
print(f"\nTiming summary saved to: {timing_path}")

  model  load_time_sec  encode_time_sec  total_time_sec
  mpnet          23.87            31.31           55.19
   stsb          25.92            30.35           56.27
     e5          14.39            31.04           45.43
  labse          38.45            29.45           67.91
finbert          14.61            38.70           53.32

Timing summary saved to: c:\madee\thesis-project\data\processed\embedding_timing.csv


### Truncation Summary

The table below records how many of the 154 texts exceed the 512-token input limit for each model-configuration-language triple. This turns the 512-token truncation limitation from a theoretical concern into a measured one, and provides the concrete numbers needed for the discussion of limitations in the methodology chapter.

In [8]:
trunc_df = pd.DataFrame(trunc_records)
trunc_df = trunc_df.round({'mean_tokens': 1, 'median_tokens': 1, 'pct_truncated': 2})

# Focus view: only rows where truncation actually occurs
print("Rows with any truncation (n_truncated > 0):")
affected = trunc_df[trunc_df['n_truncated'] > 0]
if len(affected) == 0:
    print("  None. No configuration triggered truncation for any model.")
else:
    print(affected.to_string(index=False))

print(f"\nFull table: {len(trunc_df)} rows (all 30 combinations x 2 languages)")
trunc_path = PROCESSED_DIR / "truncation_report.csv"
trunc_df.to_csv(trunc_path, index=False)
print(f"Truncation report saved to: {trunc_path}")

Rows with any truncation (n_truncated > 0):
  model  config language  n_texts  mean_tokens  median_tokens  max_tokens_untruncated  n_truncated  pct_truncated
  mpnet       2       fi      154        369.8          347.0                     723           18          11.69
  mpnet       2       en      154        333.8          318.0                     676           14           9.09
  mpnet       3       fi      154        637.3          603.0                    1091          114          74.03
  mpnet       3       en      154        595.5          579.5                    1007          103          66.88
  mpnet       5       fi      154        359.5          342.5                     679           17          11.04
  mpnet       5       en      154        340.9          315.0                     698           14           9.09
  mpnet       6       fi      154        626.8          606.0                    1050          107          69.48
  mpnet       6       en      154        611

### Verifying All 60 Embedding Files

All 60 saved files are verified to confirm they exist, carry the expected shape of (154, 768), and remain L2 normalized after being read back from disk. All three checks run as assertions so that any issue halts execution immediately rather than propagating silently to Notebook 3.

In [9]:
expected_shape = (len(df), 768)
problems = []

for model_key in models.keys():
    for config_num in configs:
        for lang in languages:
            fname = f'{model_key}_config{config_num}_{lang}.npy'
            fpath = EMBEDDINGS_DIR / fname
            if not fpath.exists():
                problems.append(f"MISSING: {fname}")
                continue
            emb = np.load(fpath)
            if emb.shape != expected_shape:
                problems.append(f"SHAPE: {fname} -> {emb.shape}")
                continue
            norms = np.linalg.norm(emb, axis=1)
            if not np.allclose(norms, 1.0, atol=1e-5):
                problems.append(
                    f"NORM: {fname} -> norms in [{norms.min():.5f}, {norms.max():.5f}]"
                )

assert not problems, "Embedding file verification failed:\n  " + "\n  ".join(problems)
print(f"All 60 embedding files verified at shape {expected_shape} and unit L2 norm.")

All 60 embedding files verified at shape (154, 768) and unit L2 norm.


### Summary

This notebook generated sentence embeddings for all 30 model-configuration combinations across five models and six text configurations, producing 60 files in total. Finnish and English texts were encoded separately, L2 normalized, shape-checked, and norm-checked before saving. FinBERT required batched manual mean pooling due to its standard BERT architecture, and is retained as a negative baseline against which the four multilingual models are compared. The e5 model received the `query: ` prefix on both sides as recommended for symmetric STS tasks by the model authors.

Two supporting tables were saved alongside the embeddings: `embedding_timing.csv` with per-model load and encode times, and `truncation_report.csv` with per-combination statistics on subword token lengths and truncation incidence. Both tables are referenced in the methodology chapter when discussing computational cost and the 512-token truncation limitation.